In [28]:
import os
os.getcwd()

'/Users/serendipity/Downloads/wu2018_replication/lasso'

In [29]:
os.getcwd()

'/Users/serendipity/Downloads/wu2018_replication/lasso'

In [9]:
# Load Labels
keys  = pd.read_csv("../keys_to_X.csv")
posts = pd.read_csv("../gendered_posts.csv")
keys_merged = pd.merge(keys, posts, on=["title_id", "post_id"], how="left")

In [10]:
# Training sample
i_train = keys_merged["training"] == 1
y = keys_merged.loc[i_train, "female"].values

In [13]:
# Load word-count matrix
wc = np.load("../X_word_count.npz", allow_pickle=True)
X = wc["X"][()]

X_train = X[i_train.values, :]

In [15]:
# OLS coefficient for each word: cov(X_j, y) / var(X_j)
import numpy as np

y = y.astype(float).ravel()
n = X_train.shape[0]

Ey = y.mean()

# column means: E[X_j]
Ex = np.asarray(X_train.mean(axis=0)).ravel()

# E[X_j^2]
Ex2 = np.asarray(X_train.power(2).mean(axis=0)).ravel()

# E[X_j y]  (sparse matrix times vector)
Exy = np.asarray((X_train.T @ y) / n).ravel()

cov = Exy - Ex * Ey
var = Ex2 - Ex**2

ols_me = np.divide(cov, var, out=np.zeros_like(cov), where=var > 0)

In [18]:
# Load vocabulary
vocab = pd.read_csv("../vocab10K.csv")
vocab["ME_OLS"] = ols_me

In [23]:
# Build Table 1
tab1_ols = pd.concat([
    vocab.sort_values("ME_OLS", ascending=False)[["word","ME_OLS"]].head(10).reset_index(drop=True),
    vocab.sort_values("ME_OLS", ascending=True)[["word","ME_OLS"]].head(10).reset_index(drop=True)
], axis=1)

print(tab1_ols)

          word    ME_OLS           word    ME_OLS
0        susan  0.768225       \xc2\xa6 -2.814264
1        she'd  0.766797            bro -0.223730
2         jane  0.764465          parag -0.222903
3      jessica  0.763879        walters -0.222881
4  girlfriends  0.761421  falsification -0.222826
5          amy  0.759020        model's -0.222785
6       ladies  0.756568      acquiring -0.222783
7        athey  0.754270         troika -0.222774
8     jennifer  0.752454            sje -0.222770
9      broette  0.750681   \xe2\x88\x9e -0.222768


In [ ]:
tab1_ols.to_csv("table1_OLS.csv", index=False)

In [25]:
mask = vocab["word"].str.match("^[a-zA-Z]+$", na=False)
vocab_clean = vocab[mask]
tab1_ols_clean = pd.concat([
    vocab_clean.sort_values("ME_OLS", ascending=False)[["word","ME_OLS"]].head(10).reset_index(drop=True),
    vocab_clean.sort_values("ME_OLS", ascending=True)[["word","ME_OLS"]].head(10).reset_index(drop=True),
], axis=1)

print(tab1_ols_clean)

          word    ME_OLS           word    ME_OLS
0        susan  0.768225            bro -0.223730
1         jane  0.764465          parag -0.222903
2      jessica  0.763879        walters -0.222881
3  girlfriends  0.761421  falsification -0.222826
4          amy  0.759020      acquiring -0.222783
5       ladies  0.756568         troika -0.222774
6        athey  0.754270            sje -0.222770
7     jennifer  0.752454          rauch -0.222767
8      broette  0.750681            jwe -0.222766
9     broettes  0.745264  overrespected -0.222766


In [26]:
tab1_ols.to_csv("table1_OLS_clean.csv", index=False)